# ND Week 5 — Support Vector Machines: Kernels and Regularization
**DX799 | Cindy Lopez**

**Question:** How well can these models predict customer conversion and campaign conversion counts? We use three datasets independently. The customer target is binary `Conversion`; the campaign targets are observed numerical `Conversions` counts, so we use regression for those two datasets. Metrics across different targets are not directly comparable.

## 1. Data loading, EDA and leakage safeguards
The uncleaned Google Ads file requires currency cleaning and missing-data treatment. Imputation, encoding and scaling are fitted only on training folds. We exclude identifiers, conversion-rate proxies, revenue and ROI from predictors to avoid outcome leakage.

In [1]:
import numpy as np, pandas as pd, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display

FILES = {
 'Customer conversion': 'digital_marketing_campaign_dataset.csv',
 'Google Ads': 'GoogleAds_DataAnalytics_Sales_Uncleaned.csv',
 'Product performance': 'marketing_and_product_performance.csv'
}
for label, filename in FILES.items():
    if not os.path.isfile(filename):
        raise FileNotFoundError(f'Missing {filename}. Place all three CSVs in the notebook folder or upload them to Colab.')
print('All three CSV files found.')
data={name:pd.read_csv(filename) for name,filename in FILES.items()}
for name, df in data.items():
    print('\n',name,'shape:',df.shape,'missing cells:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated().sum()))
    display(df.head(2))
    display(df.isna().sum().sort_values(ascending=False).head(5).to_frame('missing'))
ads=data['Google Ads'].copy()
for col in ['Cost','Sale_Amount']:
    ads[col]=pd.to_numeric(ads[col].astype(str).str.replace(r'[$,]','',regex=True),errors='coerce')
for col in ['Clicks','Impressions','Leads','Conversions','Conversion Rate']:
    ads[col]=pd.to_numeric(ads[col],errors='coerce')
# We deliberately exclude Leads, Conversion Rate and Sale_Amount as outcome-adjacent or downstream variables.
config={
 'Customer conversion':{'frame':data['Customer conversion'],'target':'ConversionRate','numeric':['Age','Income','AdSpend','ClickThroughRate','WebsiteVisits','PagesPerVisit','TimeOnSite','SocialShares','EmailOpens','EmailClicks','PreviousPurchases','LoyaltyPoints'],'categorical':['Gender','CampaignChannel','CampaignType'],'poly':['AdSpend','ClickThroughRate','TimeOnSite']},
 'Google Ads':{'frame':ads,'target':'Conversions','numeric':['Clicks','Impressions','Cost'],'categorical':['Campaign_Name','Location','Device'],'poly':['Clicks','Impressions','Cost']},
 'Product performance':{'frame':data['Product performance'],'target':'Conversions','numeric':['Budget','Clicks','Discount_Level','Bundle_Price'],'categorical':['Subscription_Tier','Common_Keywords'],'poly':['Budget','Clicks','Discount_Level']}
}
for name,c in config.items():
    print(name,'target:',c['target'],'range:',c['frame'][c['target']].min(),c['frame'][c['target']].max())

All three CSV files found.

 Customer conversion shape: (8000, 20) missing cells: 0 duplicates: 0


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1


,missing
CustomerID,0
Age,0
Gender,0
Income,0
CampaignChannel,0



 Google Ads shape: (2600, 13) missing cells: 1150 duplicates: 0


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course


,missing
Conversion Rate,626
Sale_Amount,139
Clicks,112
Cost,97
Conversions,74



 Product performance shape: (10000, 17) missing cells: 0 duplicates: 0


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative


,missing
Campaign_ID,0
Product_ID,0
Budget,0
Clicks,0
Conversions,0


Customer conversion target: ConversionRate range: 0.0100177828330717 0.1999947079999166
Google Ads target: Conversions range: 3.0 10.0
Product performance target: Conversions range: 1 999


In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error, confusion_matrix
from IPython.display import display
import matplotlib.pyplot as plt

def ohe():
    try: return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown='ignore',sparse=False)
def prep(c,scale=True):
    ns=[('impute',SimpleImputer(strategy='median'))]
    if scale: ns.append(('scale',StandardScaler()))
    return ColumnTransformer([('numeric',Pipeline(ns),c['numeric']),('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',ohe())]),c['categorical'])])

for name,c in config.items():
    target='Conversion' if name=='Customer conversion' else 'Conversions'
    y=pd.to_numeric(c['frame'][target],errors='coerce')
    print(name,'rows:',len(c['frame']),'target missing:',int(y.isna().sum()),'target median:',y.median(),'target std:',y.std())

Customer conversion rows: 8000 target missing: 0 target median: 1.0 target std: 0.3290308232678703
Google Ads rows: 2600 target missing: 74 target median: 7.0 target std: 2.2725732687366835
Product performance rows: 10000 target missing: 0 target median: 499.0 target std: 289.47949475584346


## 2. Linear and RBF SVMs
A linear kernel fits a linear decision boundary; an RBF kernel uses the kernel trick to represent nonlinear similarity without constructing every nonlinear feature explicitly. `C` controls the penalty for errors (smaller `C` gives stronger regularization). `gamma` controls the locality of the RBF similarity. Scaling is essential because SVM distances depend on feature magnitude. For computational feasibility, training uses at most 3,000 training observations per dataset; the test split remains untouched.

In [3]:
from sklearn.svm import SVC, SVR
rows=[]; details={}
for name,c in config.items():
    target='Conversion' if name=='Customer conversion' else 'Conversions'
    df=c['frame'].copy(); yy=pd.to_numeric(df[target],errors='coerce'); ok=yy.notna(); df=df.loc[ok]; yy=yy.loc[ok]
    cls=(name=='Customer conversion')
    y=yy.astype(int) if cls else yy.astype(float)
    X=df[c['numeric']+c['categorical']]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y if cls else None)
    if len(Xtr)>3000:
        Xtr,_,ytr,_=train_test_split(Xtr,ytr,train_size=3000,random_state=42,stratify=ytr if cls else None)
    for kernel in ['linear','rbf']:
        model=SVC(kernel=kernel,C=1,gamma='scale',probability=False,cache_size=500) if cls else SVR(kernel=kernel,C=1,gamma='scale',epsilon=.1,cache_size=500)
        pipe=Pipeline([('prep',prep(c)),('model',model)])
        pipe.fit(Xtr,ytr); p=pipe.predict(Xte)
        if cls:
            score=pipe.decision_function(Xte)
            row={'dataset':name,'kernel':kernel,'train_n':len(Xtr),'accuracy':accuracy_score(yte,p),'balanced_accuracy':balanced_accuracy_score(yte,p),'F1_macro':f1_score(yte,p,average='macro'),'ROC_AUC':roc_auc_score(yte,score),'minority_recall':( (p[yte==0]==0).mean() if (yte==0).any() else np.nan)}
        else:
            row={'dataset':name,'kernel':kernel,'train_n':len(Xtr),'R2':r2_score(yte,p),'RMSE':np.sqrt(mean_squared_error(yte,p)),'MAE':mean_absolute_error(yte,p)}
        rows.append(row); details[(name,kernel)]=(Xtr,Xte,ytr,yte,p,pipe)
        print(name,kernel,'completed')
results=pd.DataFrame(rows);display(results.round(4))

Customer conversion linear completed


Customer conversion rbf completed
Google Ads linear completed


Google Ads rbf completed


Product performance linear completed


Product performance rbf completed


,dataset,kernel,train_n,accuracy,balanced_accuracy,F1_macro,ROC_AUC,minority_recall,R2,RMSE,MAE
0,Customer conversion,linear,3000,0.8762,0.5000,0.4670,0.6312,0.0000,NaN,NaN,NaN
1,Customer conversion,rbf,3000,0.8856,0.5444,0.5515,0.7350,0.0909,NaN,NaN,NaN
2,Google Ads,linear,2020,NaN,NaN,NaN,NaN,NaN,-0.0094,2.2661,1.9533
3,Google Ads,rbf,2020,NaN,NaN,NaN,NaN,NaN,-0.0809,2.3450,1.9849
4,Product performance,linear,3000,NaN,NaN,NaN,NaN,NaN,-0.0061,291.8506,252.5834
5,Product performance,rbf,3000,NaN,NaN,NaN,NaN,NaN,-0.0018,291.2169,252.4386


## 3. Regularization tuning on the customer conversion task
Tune `C` and RBF `gamma` using three-fold stratified CV on the training subset only, optimizing ROC-AUC. Report the untouched test result separately. The search is intentionally compact to keep this notebook reproducible on a laptop.

In [4]:
name='Customer conversion'; c=config[name]
Xtr,Xte,ytr,yte,_,_=details[(name,'rbf')]
pipe=Pipeline([('prep',prep(c)),('model',SVC(kernel='rbf',cache_size=500))])
search=GridSearchCV(pipe,{'model__C':[.1,1,10],'model__gamma':['scale',.01]},scoring='roc_auc',cv=StratifiedKFold(3,shuffle=True,random_state=42),n_jobs=1)
search.fit(Xtr,ytr)
p=search.predict(Xte);scores=search.decision_function(Xte)
print('Best parameters:',search.best_params_,'CV ROC-AUC:',round(search.best_score_,4))
print('Test ROC-AUC:',round(roc_auc_score(yte,scores),4),'balanced accuracy:',round(balanced_accuracy_score(yte,p),4))
print('Confusion matrix:',confusion_matrix(yte,p).tolist())

Best parameters: {'model__C': 1, 'model__gamma': 0.01} CV ROC-AUC: 0.7699
Test ROC-AUC: 0.7545 balanced accuracy: 0.5
Confusion matrix: [[0, 198], [0, 1402]]


## 4. Interpretation and limitations
Compare train and test scores to identify overfitting. For imbalanced customer outcomes, overall accuracy can mask poor minority recall. ROC-AUC measures ranking quality and is not equivalent to performance at the 0.5 decision threshold. Campaign conversion counts are not the same outcome as individual customer conversion. This is predictive analysis; feature importance or model association is not causal attribution.

**Sources:** El Kharoua (2024), Nayak (2024), Shah (2024), Kaggle datasets. scikit-learn documentation: https://scikit-learn.org/stable/supervised_learning.html

**AI-use disclosure:** ChatGPT assisted with notebook code, structure and explanation. All quantitative claims should be checked against the saved executed output.

## 5. Model complexity: training versus test performance

A model can perform well on training observations but generalize poorly. Here I compare training and held-out test performance for both kernels across all three datasets. Because the campaign datasets predict **conversion counts**, their regression metrics must not be compared numerically to the classification metrics from the customer dataset.

In [ ]:
# Compare model fit on the training sample and the untouched test sample.
complexity_rows = []
for (dataset, kernel), (Xtr, Xte, ytr, yte, pred, pipe) in details.items():
    train_pred = pipe.predict(Xtr)
    if dataset == 'Customer conversion':
        complexity_rows.append({
            'dataset': dataset, 'kernel': kernel,
            'train_balanced_accuracy': balanced_accuracy_score(ytr, train_pred),
            'test_balanced_accuracy': balanced_accuracy_score(yte, pred),
            'train_macro_F1': f1_score(ytr, train_pred, average='macro'),
            'test_macro_F1': f1_score(yte, pred, average='macro')
        })
    else:
        complexity_rows.append({
            'dataset': dataset, 'kernel': kernel,
            'train_R2': r2_score(ytr, train_pred),
            'test_R2': r2_score(yte, pred),
            'train_RMSE': np.sqrt(mean_squared_error(ytr, train_pred)),
            'test_RMSE': np.sqrt(mean_squared_error(yte, pred))
        })
complexity_table = pd.DataFrame(complexity_rows)
display(complexity_table.round(4))
print('Interpretation: large train–test gaps suggest overfitting; weak scores on both sets suggest underfitting or limited signal.')

## 6. How `C` and `gamma` change validation performance

This chart shows the three-fold cross-validation ROC-AUC from the **training-only** grid search. Higher `C` penalizes classification errors more strongly; higher numeric `gamma` makes the RBF similarity more localized. A better training fit does not automatically mean better test performance.

In [ ]:
# Plot the cross-validation tuning results; these are NOT test-set scores.
cv_results = pd.DataFrame(search.cv_results_)
cv_results['C'] = cv_results['param_model__C'].astype(float)
cv_results['gamma_label'] = cv_results['param_model__gamma'].astype(str)
pivot = cv_results.pivot(index='C', columns='gamma_label', values='mean_test_score')
display(pivot.round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
for gamma_label in pivot.columns:
    ax.plot(pivot.index, pivot[gamma_label], marker='o', label=f'gamma={gamma_label}')
ax.set_xscale('log')
ax.set_xlabel('Regularization parameter C (log scale)')
ax.set_ylabel('Mean cross-validation ROC-AUC')
ax.set_title('RBF SVM: tuning C and gamma')
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## 7. Classification diagnostics: ROC curve and confusion matrix

The ROC curve summarizes discrimination across thresholds, while the confusion matrix shows errors at the model's default decision boundary. The customer dataset is imbalanced, so I also report class-specific recall and a precision–recall curve for the **non-conversion** class (label 0). The latter matters if the business question is identifying customers unlikely to convert.

In [ ]:
from sklearn.metrics import (
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay, classification_report
)

# Held-out test observations only
Xtr, Xte, ytr, yte, _, _ = details[('Customer conversion', 'rbf')]
tuned_pred = search.predict(Xte)
tuned_score = search.decision_function(Xte)
print('Classification report (0 = non-conversion; 1 = conversion):')
print(classification_report(yte, tuned_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(yte, tuned_pred, labels=[0, 1], ax=ax, cmap='Blues')
ax.set_title('Tuned RBF SVM — held-out test')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(yte, tuned_score)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(yte, tuned_score):.3f}')
ax.plot([0, 1], [0, 1], '--', label='Random ranking')
ax.set(xlabel='False positive rate', ylabel='True positive rate',
       title='Tuned RBF SVM — ROC curve')
ax.legend()
plt.tight_layout()
plt.show()

# The SVC decision score increases for class 1, so negate it for class 0.
nonconversion = (np.asarray(yte) == 0).astype(int)
nonconversion_score = -np.asarray(tuned_score)
precision, recall, _ = precision_recall_curve(nonconversion, nonconversion_score)
ap = average_precision_score(nonconversion, nonconversion_score)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(recall, precision, label=f'Average precision = {ap:.3f}')
ax.axhline(nonconversion.mean(), linestyle='--', label='Non-conversion prevalence')
ax.set(xlabel='Recall (non-conversion)', ylabel='Precision (non-conversion)',
       title='Precision–recall curve for non-conversions')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Interpretation for Milestone One

After executing the cells above, answer these questions using the **observed numbers**:

1. Which kernel had the better held-out score for each dataset? Were the differences substantial?
2. Which model shows the largest train–test gap? Does that indicate overfitting?
3. Which `C` and `gamma` combination produced the best cross-validation ROC-AUC, and how did it perform on the held-out test set?
4. How well did the tuned classifier identify **non-converters**, rather than simply predicting the majority class?
5. Would the model be useful for allocating advertising budget? Explain what can and cannot be inferred from predictive associations.

**Limitations:** The SVM analysis trains on at most 3,000 training observations per dataset for computational practicality. The held-out test sets remain untouched, but the training subset may not capture every pattern. SVM performance depends on scaling, tuning, and the selected features. Conversion counts and individual conversion labels represent different prediction problems.

**Reproducibility:** Run all cells from top to bottom in VS Code with the three CSV files available. Save the notebook after execution to preserve outputs. This enhanced section is newly added and must be executed before its findings can be cited in the final report.